# Загрузка библиотек

In [34]:
import pandas as pd
import psycopg2
import warnings
from typing import List, Tuple
from psycopg2.extras import execute_values
import sqlite3

warnings.filterwarnings('ignore')

# PostgreSQL

## Подключение к базе данных

In [2]:
def get_conn(dbname, user, password, host, port):
    return psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)

In [28]:
dbname = "ETL" # Название базы данных
user = "postgres" # Имя пользователя для подключения
password = "86754231qaZ" # Пароль пользователя для подключения
host = "localhost" # Хост
port = "5432" # Порт

conn = get_conn(dbname, user, password, host, port)

## Чтение данных в pandas DataFrame

In [4]:
data = pd.read_sql("SELECT * FROM movies", con=conn)

data

,id,name,description,year,rating,votes,movie_length,age_rating,genres,countries
0,5921398,Гатчина. Молчание Сильвии,История 867 дней фашистской оккупации Гатчины ...,2024,8.637,3531,0,18,документальный|военный,Россия
1,5613191,Цвета зла: Красный,На пляже находят тело молодой девушки. Расслед...,2024,6.253,439,111,0,триллер|драма|криминал|детектив,Польша
2,5611775,nan,nan,2024,6.460,115,101,0,триллер|драма|криминал,Испания|Бельгия|Франция
3,5461825,Что видят животные,Как животные воспринимают окружающий мир? Как ...,2018,7.889,746,28,6,короткометражка|реальное ТВ,США
4,5595677,Тастаймын-ау сени,Жарас и Алмагуль женаты уже более десяти лет. ...,2024,6.853,16508,90,18,комедия|ужасы,Казахстан
...,...,...,...,...,...,...,...,...,...,...
7795,1065458,nan,nan,1999,7.209,263,4,0,комедия|музыка|короткометражка,США
7796,15367,Врата безмолвия,Преуспевающий бизнесмен Мелвин Деверо по пути ...,1991,5.847,283,87,18,триллер|детектив,Италия
7797,296375,Создание «Триллера»,nan,1983,8.367,404,70,0,документальный|музыка,США
7798,4529471,nan,nan,2020,7.880,81,4,0,музыка,США


## Чтение данных с опциональными фильтрами

In [5]:
def get_data_with_filters(year=None, rating=None, votes=None, countries=None):
    query = "SELECT * FROM movies WHERE true"

    if year:
        query += f" AND year = {year}"
    if rating:
        query += f" AND rating > {rating}"
    if votes:
        query += f" AND votes > {votes}"
    if countries:
        query += f" AND countries LIKE '{countries}'"
    
    data = pd.read_sql(query, con=conn)

    return data

In [6]:
data = get_data_with_filters(year=2020, rating=7, votes=300, countries="США")

data

,id,name,description,year,rating,votes,movie_length,age_rating,genres,countries
0,1337514,Натали Вуд: Что остается за кадром,История жизни легендарной актрисы глазами ее б...,2020,7.548,488,100,18,документальный,США
1,1361177,Схема,Документальный фильм восполняет очередной проб...,2020,7.353,351,68,18,документальный|криминал,США
2,1332684,Вам не убить Дэвида Аркетта,Мир рестлинга назвал актера Дэвида Аркетта сам...,2020,7.237,974,91,18,документальный|биография|спорт,США
3,1203057,Воздушная гимнастка,"Джейн, воздушная гимнастка, готовится к танцев...",2020,7.030,1667,102,18,драма,США
4,802295,Дети шоу-бизнеса,nan,2020,7.166,1827,95,18,документальный,США
5,1411901,Спасти планету,"Земле, нашей любимой планете и единственному д...",2020,8.270,2281,89,6,документальный,США
6,4345981,Носильщики: Нерассказанная история на Эвересте,Молодой привилегированный американец пытается ...,2020,7.294,2017,55,18,документальный|приключения,США
7,1331982,"Ништяк, браток",В середине 2000-х художник Мэтт Фьюри рисовал ...,2020,7.511,3974,92,18,документальный|комедия,США
8,1395414,"Безумен, но не болен",Хроника наблюдений режиссера за работой психол...,2020,7.357,6615,119,18,документальный|криминал|биография,США
9,1398862,Мечта Робина,"В августе 2014 года мир был потрясен, узнав, ч...",2020,8.111,13223,77,18,документальный|биография,США


## Добавление данных в базу данных

### Чтение данных из файла

In [7]:
filepath = "data\Data.xlsx" # Путь до файла
sheet_name = "Sheet_2" # Название листа в excel файле
header = 0 # Индекс строки, которую принять за названия колонок

data = pd.read_excel(filepath, sheet_name=sheet_name, header=header)

data

,id_2,name_2
0,1,John
1,2,Jerry
2,3,Joe
3,4,Julia
4,5,Jessy


### Добавление данных в базу данных

In [ ]:
def create_table(query):
    """
    Создание таблицы в базе данных
    """
    try:
        cur = conn.cursor()
        cur.execute(query)
        conn.commit()

    except Exception as e:
        print("Ошибка:", e)
        conn.close()

In [33]:
def insert_data(query, data: List[Tuple]):
    """
    Добавление данных в таблицу из базы данных
    """
    try:
        cur = conn.cursor()
        execute_values(cur, query, data)
        conn.commit()

    except Exception as e:
        print("Ошибка:", e)
        conn.close()

In [15]:
query = """CREATE TABLE IF NOT EXISTS excel_table (
    id SERIAL PRIMARY KEY,
    name VARCHAR NOT NULL
);"""

create_table(query)

In [30]:
query = """INSERT INTO "excel_table" (id, name)
        VALUES %s"""

insert_data(query, data.values.tolist())

# SQLite

## Подключение к базе данных

In [36]:
filepath = "db/database.db" # Путь до файла с базой данных

conn = sqlite3.connect(filepath) 

## Добавление данных в базу данных

### Чтение данных из файла

In [37]:
filepath = "data\Data.xlsx" # Путь до файла
sheet_name = "Sheet_2" # Название листа в excel файле
header = 0 # Индекс строки, которую принять за названия колонок

data = pd.read_excel(filepath, sheet_name=sheet_name, header=header)

data

,id_2,name_2
0,1,John
1,2,Jerry
2,3,Joe
3,4,Julia
4,5,Jessy


### Добавление данных в базу данных

In [38]:
table_name = "excel_table" # Название новой таблицы
if_exists = "replace" # Поведение, если таблица уже существует
index = False # Сохранять ли индекс в отдельную колонку

data.to_sql(name=table_name, con=conn, if_exists=if_exists, index=index)

5

## Чтение данных в pandas DataFrame

In [39]:
data = pd.read_sql("SELECT * FROM excel_table", con=conn)

data

,id_2,name_2
0,1,John
1,2,Jerry
2,3,Joe
3,4,Julia
4,5,Jessy


## Чтение данных с опциональными фильтрами

In [40]:
def get_data_with_filters(id_2=None, name_2=None):
    query = "SELECT * FROM excel_table WHERE true"

    if id_2:
        query += f" AND id_2 > {id_2}"
    if name_2:
        query += f" AND name_2 LIKE '{name_2}'"
    
    data = pd.read_sql(query, con=conn)

    return data

In [42]:
data = get_data_with_filters(id_2=2, name_2="Joe")

data

,id_2,name_2
0,3,Joe
